# 🏨 Hotel Curator — Advanced ML Experimentation

To ensure our modeling is rigorous, detailed-oriented, and explores the entire algorithm space, this notebook trains and compares multiple advanced model architectures including Gradient Boosting, XGBoost, and Multi-Layer Perceptrons (Neural Networks).

We also introduce Hyperparameter Tuning (`GridSearchCV`) and ROC-AUC evaluation to deeply analyze model performance beyond simple accuracy.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report
import warnings
warnings.filterwarnings('ignore')

# 1. Load Data
df = pd.read_csv('data/labeled_clauses.csv')
attrs = ['cleanliness', 'staff_service', 'wifi_quality', 'noise_level', 'location']
print(f"Loaded {len(df):,} labeled clauses.")

Loaded 203,729 labeled clauses.


In [2]:
# 2. Advanced Feature Engineering
# We use up to 3-grams to capture complex phrases and increase features to 25,000.
vectorizer = TfidfVectorizer(max_features=25000, stop_words='english', ngram_range=(1, 3))
X = vectorizer.fit_transform(df['clause'].fillna(''))
print("Advanced Vectorization Complete! Shape:", X.shape)

Advanced Vectorization Complete! Shape: (203729, 25000)


In [ ]:
# 3. Deep Model Benchmark
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier

# Map string labels to binary integers for models like XGBoost
label_map = {'negative': 0, 'positive': 1}

advanced_models = {
    "Linear SVM (L2 Reg)": LinearSVC(C=0.5, class_weight='balanced', max_iter=2000),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3),
    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric='logloss', scale_pos_weight=1.5),
    "MLP Neural Net": MLPClassifier(hidden_layer_sizes=(50,), max_iter=20, early_stopping=True)
}

for attr in attrs:
    print(f"\n{'='*50}")
    print(f"🧪 DEEP BENCHMARK: {attr.upper()}")
    print(f"{'='*50}")
    
    mask = df[attr] != 'not_mentioned'
    y_str = df.loc[mask, attr]
    y_binary = y_str.map(label_map)
    X_subset = X[mask.values]
    
    if len(y_binary) < 50:
        print("Not enough data.")
        continue
        
    X_train, X_test, y_train, y_test = train_test_split(X_subset, y_binary, test_size=0.15, random_state=42)
    
    best_roc = 0
    best_name = ""
    
    for name, clf in advanced_models.items():
        # Train model
        clf.fit(X_train, y_train)
        
        # Predict
        preds = clf.predict(X_test)
        acc = accuracy_score(y_test, preds)
        
        # ROC-AUC calculation (handling models that don't output probas naturally)
        if hasattr(clf, "predict_proba"):
            probs = clf.predict_proba(X_test)[:, 1]
            roc = roc_auc_score(y_test, probs)
        else:
            decision = clf.decision_function(X_test)
            roc = roc_auc_score(y_test, decision)
            
        print(f"  → {name:<25} | Accuracy: {acc*100:.1f}% | ROC-AUC: {roc:.3f}")
        
        if roc > best_roc:
            best_roc = roc
            best_name = name
            
    print(f"\n  🏆 WINNER (By ROC-AUC) for {attr.upper()}: {best_name}")


🧪 DEEP BENCHMARK: CLEANLINESS
  → Linear SVM (L2 Reg)       | Accuracy: 83.2% | ROC-AUC: 0.912
  → Gradient Boosting         | Accuracy: 79.5% | ROC-AUC: 0.881
  → XGBoost                   | Accuracy: 80.7% | ROC-AUC: 0.890
  → MLP Neural Net            | Accuracy: 83.7% | ROC-AUC: 0.916

  🏆 WINNER (By ROC-AUC) for CLEANLINESS: MLP Neural Net

🧪 DEEP BENCHMARK: STAFF_SERVICE
  → Linear SVM (L2 Reg)       | Accuracy: 80.4% | ROC-AUC: 0.887
  → Gradient Boosting         | Accuracy: 75.6% | ROC-AUC: 0.832
  → XGBoost                   | Accuracy: 77.8% | ROC-AUC: 0.868
  → MLP Neural Net            | Accuracy: 81.6% | ROC-AUC: 0.889

  🏆 WINNER (By ROC-AUC) for STAFF_SERVICE: MLP Neural Net

🧪 DEEP BENCHMARK: WIFI_QUALITY
  → Linear SVM (L2 Reg)       | Accuracy: 73.3% | ROC-AUC: 0.806


In [ ]:
# 4. Hyperparameter Tuning (GridSearchCV)
print("\nRunning Hyperparameter Grid Search on the best algorithm (Linear SVM) for Cleanliness...")
attr = 'cleanliness'
mask = df[attr] != 'not_mentioned'
y_binary = df.loc[mask, attr].map(label_map)
X_subset = X[mask.values]
X_train, X_test, y_train, y_test = train_test_split(X_subset, y_binary, test_size=0.15, random_state=42)

param_grid = {
    'C': [0.1, 0.5, 1.0, 5.0],
    'loss': ['hinge', 'squared_hinge']
}

grid = GridSearchCV(LinearSVC(class_weight='balanced', max_iter=2000), param_grid, cv=3, scoring='roc_auc', n_jobs=-1)
grid.fit(X_train, y_train)

print(f"Best Parameters Found: {grid.best_params_}")
print(f"Best Cross-Validated ROC-AUC: {grid.best_score_:.3f}")
print("Detailed-Oriented Modeling Phase Complete! ✅")